# Dự đoán sống sót

## Ôn tập (preview)

## Khởi tạo thí nghiệm

### Khai báo thư viện

In [4]:
#Imports
import warnings
warnings.filterwarnings('ignore')

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from IPython import display
import joblib

from scipy.stats import spearmanr
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.ensemble import VotingClassifier, StackingClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, roc_auc_score

### Tham số thực nghiệm

In [5]:
params_cfg = {
    # "action"   : "train_feat01",  
    # "feat_path": "../../exps/featbase_251028/data.npz",
    "seed"    : 42, # Set random seed
    "exp_dir" : os.path.abspath('../exps/output'),
    "model_dir": os.path.abspath('../exps/gridsearchcv'),
    'exp_name': 'result_gridsearchcv',
    "data_dir": os.path.abspath("../exps/feature1"),
    "verbose" : True,
    "k_fold": 10,
}
params_cfg.update(**{
    "save_dir": os.path.abspath(f'{params_cfg["exp_dir"]}/{params_cfg["exp_name"]}')
})

for v in params_cfg:
    print(f'+ {v}: {params_cfg[v]}')

globals().update(**params_cfg)

+ seed: 42
+ exp_dir: d:\ML_git\Machine_Learning_basic\Titanic_final\exps\output
+ model_dir: d:\ML_git\Machine_Learning_basic\Titanic_final\exps\gridsearchcv
+ exp_name: result_gridsearchcv
+ data_dir: d:\ML_git\Machine_Learning_basic\Titanic_final\exps\feature1
+ verbose: True
+ k_fold: 10
+ save_dir: d:\ML_git\Machine_Learning_basic\Titanic_final\exps\output\result_gridsearchcv


### Load dữ liệu đã tiền xử lí 

In [6]:
df_train = pd.read_excel(f'{params_cfg["data_dir"]}/train_df_preprocess.xlsx')
df_test = pd.read_excel(f'{params_cfg["data_dir"]}/test_df_preprocess.xlsx')

data = np.load(f'{params_cfg["data_dir"]}/feat_preprocess.npz', allow_pickle=True)

# Load feature columns
feat_cols = np.load(f'{params_cfg["data_dir"]}/feature_columns.npz', allow_pickle=True)
feature_columns = feat_cols['feature_columns']

# Chuyển về DataFrame
x= pd.DataFrame(data['x_train'], columns=feature_columns)
y = pd.Series(data['y_train'], name='Survived')
x_test_final = pd.DataFrame(data['x_test'], columns=feature_columns)

display.display(df_train.sample(5))
if params_cfg["verbose"]:
    print("-"*10, "information", "-"*10)
    print(f'train shape: {df_train.shape}')
    print(f'test shape: {df_test.shape}')
    print(f'train-col: {set(df_train.columns)}')
    print(f'test-col: {set(df_test.columns)}')
    print("Union:", set(df_train.columns).intersection(set(df_test.columns)))
    print("Difference:", set(df_train.columns).difference(set(df_test.columns)))

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,...,Embarked,Title,FamilySize,IsChild,IsMother,HasCabin,Age*Pclass,Deck,Fare_Pclass,TicketPrefix
845,846,0,3,"Abbing, Mr. Anthony",male,3.761200,0,0,C.A. 5547,2.145931,...,S,Mr,1,0,0,0,11.283600,U,0.715310,C
72,73,0,2,"Hood, Mr. Ambrose Jr",male,3.091042,0,0,S.O.C. 14879,4.310799,...,S,Mr,1,0,0,0,6.182085,U,2.155400,S
116,117,0,3,"Connors, Mr. Patrick",male,4.269697,0,0,370369,2.169054,...,Q,Mr,1,0,0,0,12.809092,U,0.723018,NUM
190,191,1,2,"Pinsky, Mrs. (Rosa)",female,3.496508,0,0,234604,2.639057,...,S,Mrs,1,0,0,0,6.993015,U,1.319529,NUM
625,626,0,1,"Sutton, Mr. Frederick",male,4.127134,0,0,36963,3.506182,...,S,Mr,1,0,0,1,4.127134,D,3.506182,NUM


---------- information ----------
train shape: (891, 21)
test shape: (418, 20)
train-col: {'Name', 'FamilySize', 'Survived', 'Fare', 'SibSp', 'Age*Pclass', 'Fare_Pclass', 'Embarked', 'PassengerId', 'TicketPrefix', 'IsMother', 'Sex', 'HasCabin', 'Age', 'Deck', 'Pclass', 'Cabin', 'Parch', 'Title', 'Ticket', 'IsChild'}
test-col: {'Name', 'FamilySize', 'Fare', 'SibSp', 'Age*Pclass', 'Fare_Pclass', 'Embarked', 'PassengerId', 'TicketPrefix', 'IsMother', 'Sex', 'HasCabin', 'Age', 'Deck', 'Pclass', 'Cabin', 'Parch', 'Title', 'Ticket', 'IsChild'}
Union: {'Name', 'FamilySize', 'Fare', 'SibSp', 'Age*Pclass', 'Fare_Pclass', 'Embarked', 'PassengerId', 'TicketPrefix', 'IsMother', 'Sex', 'HasCabin', 'Age', 'Deck', 'Pclass', 'Cabin', 'Parch', 'Title', 'Ticket', 'IsChild'}
Difference: {'Survived'}


### Preprocess pipeline

In [7]:
# Preprocessor (ColumnTransformer)
num_features = [
    'Age','Fare','FamilySize','Fare_Pclass','Age*Pclass'
]
cat_features = [
    'Pclass','Sex','Embarked','Title','IsChild','IsMother',
    'Deck','HasCabin','TicketPrefix'
]

num_transformer = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
cat_transformer = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))])

preprocessor = ColumnTransformer([('num', num_transformer, num_features), ('cat', cat_transformer, cat_features)])

## Models Training

**Các mô hình dùng để phân tích**: `Logistic Regression`, `Random Forest`, `XGBoost`, `SVM`

Định nghĩa các pipeline mô hình học máy (dùng chung preprocessor, khác models)
- `Logistic Regression`: mô hình tuyến tính cơ bản (baseline)
- `Random Forest`: tập hợp nhiều cây quyết định, giúp giảm overfitting
- `XGBoost`: mô hình boosting mạnh mẽ, cho hiệu suất cao nhất
- `SVM`: bộ phân loại phi tuyến, phù hợp với ranh giới phức tạp

In [8]:
# --- Khởi tạo các pipeline ---
models = {
    'Logistic Regression': Pipeline([
        ('preprocessor', preprocessor),
        ('model', LogisticRegression(max_iter=1000,random_state=42,C=1.0,solver='lbfgs'))
    ]),

    'Random Forest': Pipeline([
        ('preprocessor', preprocessor),
        ('model', RandomForestClassifier(random_state=42, n_jobs=-1,n_estimators=500, max_depth=5,min_samples_split=5, min_samples_leaf=3))
    ]),

    'XGBoost': Pipeline([
        ('preprocessor', preprocessor),
        ('model', XGBClassifier(random_state=42,eval_metric='logloss',n_jobs=-1,n_estimators=120,max_depth=2,learning_rate=0.08,subsample=0.6,colsample_bytree=0.6,reg_lambda=3,reg_alpha=2))
    ]),

    'SVM': Pipeline([
        ('preprocessor', preprocessor),
        ('model', SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42))
    ])
}

In [9]:
tunning_models = {
    'Random Forest': Pipeline([
        ('preprocessor', preprocessor),
        ('model', RandomForestClassifier(random_state=42, n_jobs=-1,n_estimators=500, max_depth=5,min_samples_split=5, min_samples_leaf=3))
    ]),

    'SVM': Pipeline([
        ('preprocessor', preprocessor),
        ('model', SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42))
    ])
}

- Mỗi mô hình đều kết hợp với cùng bộ xử lý dữ liệu (preprocessor) để đảm bảo đầu vào nhất quán.
- Lưu trong dictionary models giúp dễ huấn luyện và so sánh kết quả giữa các thuật toán.

### Đánh giá từng model bằng cross-validation

#### Baseline 

In [10]:
cv = StratifiedKFold(n_splits=params_cfg["k_fold"], shuffle=True, random_state=42)

results = {'Model': [], 'Metric': [], 'Score': []}

for name, model in models.items():
    acc_scores = cross_val_score(model, x, y, cv=cv, scoring='accuracy')
    f1_scores  = cross_val_score(model, x, y, cv=cv, scoring='f1')
    auc_scores = cross_val_score(model, x, y, cv=cv, scoring='roc_auc')

    # Lưu chi tiết từng lần CV
    for s in acc_scores:
        results['Model'].append(name)
        results['Metric'].append('Accuracy')
        results['Score'].append(s)
    for s in f1_scores:
        results['Model'].append(name)
        results['Metric'].append('F1 Score')
        results['Score'].append(s)
    for s in auc_scores:
        results['Model'].append(name)
        results['Metric'].append('ROC AUC')
        results['Score'].append(s)

    # Tính trung bình và độ lệch chuẩn
    acc_mean, acc_std = acc_scores.mean(), acc_scores.std()
    f1_mean, f1_std = f1_scores.mean(), f1_scores.std()
    auc_mean, auc_std = auc_scores.mean(), auc_scores.std()

    print(f"{name} CV Results:")
    print(f"  Accuracy: {acc_mean:.4f} ± {acc_std:.4f}")
    print(f"  F1 Score: {f1_mean:.4f} ± {f1_std:.4f}")
    print(f"  ROC AUC:  {auc_mean:.4f} ± {auc_std:.4f}")
    print("-" * 40)

Logistic Regression CV Results:
  Accuracy: 0.8192 ± 0.0338
  F1 Score: 0.7568 ± 0.0479
  ROC AUC:  0.8750 ± 0.0377
----------------------------------------
Random Forest CV Results:
  Accuracy: 0.8271 ± 0.0321
  F1 Score: 0.7529 ± 0.0557
  ROC AUC:  0.8710 ± 0.0459
----------------------------------------
XGBoost CV Results:
  Accuracy: 0.8159 ± 0.0365
  F1 Score: 0.7477 ± 0.0587
  ROC AUC:  nan ± nan
----------------------------------------
SVM CV Results:
  Accuracy: 0.8350 ± 0.0227
  F1 Score: 0.7652 ± 0.0406
  ROC AUC:  0.8727 ± 0.0339
----------------------------------------


- Sử dụng StratifiedKFold (10-fold CV) để đánh giá 4 mô hình khác nhau.
- Tính 3 chỉ số: Accuracy, F1 Score, ROC AUC cho từng mô hình.
- Lưu và in kết quả trung bình ± độ lệch chuẩn để so sánh hiệu suất và độ ổn định giữa các mô hình.

#### Tinh chỉnh tham số (GridSearchCV)

In [11]:
param_grids = {
    'Random Forest': {
        'model__n_estimators': [100, 300, 500],      # số lượng cây
        'model__max_depth': [3, 5, 10, None],         # độ sâu tối đa
        'model__min_samples_split': [2, 5, 10],           # min số mẫu để chia node
        'model__min_samples_leaf': [1, 2, 4],             # min mẫu ở leaf
        'model__max_features': ['sqrt', 'log2', None],    # cách chọn feature
    },
    'SVM': {
        'model__C': [0.1, 1.0, 100],              # regularization strength
        'model__kernel': ['linear', 'rbf', 'poly'],
        'model__gamma': ['scale', 'auto', 0.01, 0.1, 1],  # hệ số kernel
        'model__degree': [2, 3],                       # bậc của kernel poly
    }
}

In [12]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

best_models = {}
results = []

# --- GridSearchCV cho từng mô hình ---
for name, model in tunning_models.items():
    print(f"\n Đang chạy GridSearchCV cho {name} ...")

    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grids[name],
        cv=cv,
        scoring='roc_auc',
        n_jobs=-1,
        verbose=1
    )

    grid.fit(x, y)
    joblib.dump(grid.best_estimator_, f'{params_cfg["model_dir"]}/model_{name}.joblib')   
    best_models[name] = grid.best_estimator_

    print(f" {name} - Best Params: {grid.best_params_}")
    print(f"   Best ROC AUC: {grid.best_score_:.4f}")



 Đang chạy GridSearchCV cho Random Forest ...
Fitting 10 folds for each of 324 candidates, totalling 3240 fits


KeyboardInterrupt: 

## Xuất ra file kết quả

In [ ]:
# Tạo thư mục lưu kết quả
os.makedirs(params_cfg["save_dir"], exist_ok=True)

submissions = {}

# Train và dự đoán cho từng model (sử dụng best_models đã được tune)
for name, model in best_models.items():
    print(f"Training {name} (Tuned)...")
    
    # Dự đoán trên test set
    predictions = model.predict(x_test_final)
    
    # Lưu vào dictionary với tên file phù hợp
    filename = f'submission_{name.lower().replace(" ", "_")}_tuned.csv'
    submissions[filename] = predictions
    
    print(f"✓ {name} (Tuned) training completed!")
    print("-" * 40)

# Lưu tất cả submissions ra file CSV
print("\n📁 Saving submission files...")
for filename, predictions in submissions.items():
    submission = pd.DataFrame({
        'PassengerId': df_test['PassengerId'],
        'Survived': predictions.astype(int)
    })

    filepath = os.path.join(params_cfg["save_dir"], filename)
    submission.to_csv(filepath, index=False)
    print(f"✓ Saved: {filepath}")

print(f"\n🎉 Created {len(submissions)} submission files in: {params_cfg['save_dir']}")

# Tùy chọn: So sánh kết quả baseline vs tuned
print("\n📊 Comparing predictions...")
for name in best_models.keys():
    baseline_file = f'submission_{name.lower().replace(" ", "_")}.csv'
    tuned_file = f'submission_{name.lower().replace(" ", "_")}_tuned.csv'
    
    if baseline_file in submissions and tuned_file in submissions:
        baseline_pred = submissions[baseline_file]
        tuned_pred = submissions[tuned_file]
        diff_count = (baseline_pred != tuned_pred).sum()
        print(f"{name}: {diff_count} predictions changed after tuning")

Training Logistic Regression...
✓ Logistic Regression training completed!
----------------------------------------
Training Random Forest...
✓ Random Forest training completed!
----------------------------------------
Training XGBoost...
✓ XGBoost training completed!
----------------------------------------
Training SVM...
✓ SVM training completed!
----------------------------------------

📁 Saving submission files...
✓ Saved: d:\ML_git\Machine_Learning_basic\Titanic_final\exps\output\result_baseline\submission_logistic_regression.csv
✓ Saved: d:\ML_git\Machine_Learning_basic\Titanic_final\exps\output\result_baseline\submission_random_forest.csv
✓ Saved: d:\ML_git\Machine_Learning_basic\Titanic_final\exps\output\result_baseline\submission_xgboost.csv
✓ Saved: d:\ML_git\Machine_Learning_basic\Titanic_final\exps\output\result_baseline\submission_svm.csv

🎉 Created 4 submission files in: d:\ML_git\Machine_Learning_basic\Titanic_final\exps\output\result_baseline
